# Mapeamento eleitoral de Divinópolis – Eleição presidencial 2026


Dados de entrada (gerados pelo `recorte_divinopolis.py` a partir dos arquivos oficiais do TSE):
- `dados/votacao_presidente_divi.csv`: votos para presidente por seção, 1º e 2º turnos
- `dados/locais_votacao_divi.csv`: locais de votação de cada seção, com latitude e longitude

## 1. Bibliotecas e leitura dos dados

Lemos tudo como texto (`dtype=str`) para não perder nada na leitura. Depois convertemos só as colunas numéricas que vamos usar.

In [26]:
import pandas as pd

votos = pd.read_csv("dados/votacao_presidente_divi.csv", sep=";", dtype=str)
locais = pd.read_csv("dados/locais_votacao_divi.csv", sep=";", dtype=str)

votos["QT_VOTOS"] = votos["QT_VOTOS"].astype(int)

print("votos:", votos.shape, "| locais:", locais.shape)

votos: (6144, 26) | locais: (1008, 41)


## Funções e definições usadas nos dois turnos

In [27]:
import numpy as np
import folium
import geopandas as gpd

# Colunas do arquivo de locais que levamos para a tabela de votos no cruzamento
colunas_local = ["NR_ZONA", "NR_SECAO", "NR_LOCAL_VOTACAO", "NM_LOCAL_VOTACAO",
                 "DS_ENDERECO", "NM_BAIRRO", "NR_LATITUDE", "NR_LONGITUDE"]


def juntar_textos(valores):
    """Junta textos diferentes sem repetir (ex.: dois nomes do mesmo prédio)."""
    unicos = [v for v in dict.fromkeys(valores) if isinstance(v, str) and v.strip()]
    return " / ".join(unicos)


def pct_br(valor):
    """57.6 -> '57,6%' (vírgula decimal, padrão brasileiro)."""
    return f"{valor:.1f}%".replace(".", ",")


def num_br(valor):
    """2074 -> '2.074' (ponto como separador de milhar, padrão brasileiro)."""
    return f"{valor:,}".replace(",", ".")


# Opacidade do mapa de fundo. Ajuste olhando o resultado.
OPACIDADE_FUNDO = 0.6


ESTILO_TOOLTIP = """
    background-color: #ffffff;
    color: #222222;
    font-family: 'Helvetica Neue', Arial, sans-serif;
    font-size: 13px;
    border-radius: 8px;
    border: 1px solid #cccccc;
    padding: 10px 12px;
    box-shadow: 3px 3px 10px rgba(0, 0, 0, 0.25);
"""


def mapa_base():
    """Mapa vazio com o fundo usado em todos os mapas (1º e 2º turnos)."""
    m = folium.Map(tiles=None)
    folium.TileLayer(
        tiles="https://tile.openstreetmap.de/{z}/{x}/{y}.png",
        attr="&copy; OpenStreetMap contributors",
        name="Mapa base (OpenStreetMap em cinza)",
        opacity=OPACIDADE_FUNDO,
        class_name="fundo-cinza",     
    ).add_to(m)
    m.get_root().header.add_child(folium.Element(
        "<style>"
        ".fundo-cinza { filter: grayscale(100%); }"
        ".leaflet-container { background: #ffffff; }"   # fundo branco atrás dos tiles 
        "</style>"
    ))
    return m

## 1º turno – preparação dos dados


O caminho é o mesmo do 2º turno (explicado em detalhe nos Passos 1 a 5 do 2º turno, mais abaixo):
filtrar o turno → pivotar → cruzar com os locais pela chave `NR_ZONA` + `NR_SECAO` → agrupar por coordenada.
A diferença é que no 1º turno há **13 candidatos**, e não 2.

### Ranking no município

Os "principais candidatos" são os **4 mais votados no município** 


In [28]:
votos_1t = votos[votos["NR_TURNO"] == "1"].copy()
locais_1t = locais[locais["NR_TURNO"] == "1"].copy()

BRANCO_NULO = ["VOTO BRANCO", "VOTO NULO"]

# Ranking da cidade
ranking_divi = (
    votos_1t[~votos_1t["NM_VOTAVEL"].isin(BRANCO_NULO)]
    .groupby("NM_VOTAVEL")["QT_VOTOS"].sum()
    .sort_values(ascending=False)
    .reset_index()
)
ranking_divi["PCT_DIVI"] = 100 * ranking_divi["QT_VOTOS"] / ranking_divi["QT_VOTOS"].sum()
ranking_divi["POS_DIVI"] = range(1, len(ranking_divi) + 1)

ranking_divi.round(1)

,NM_VOTAVEL,QT_VOTOS,PCT_DIVI,POS_DIVI
0,FLAVIO NANTES BOLSONARO,71478,54.0,1
1,LUIZ INÁCIO LULA DA SILVA,49403,37.3,2
2,AUGUSTO JORGE CURY,4568,3.4,3
3,RENAN ANTONIO FERREIRA DOS SANTOS,3219,2.4,4
4,RONALDO RAMOS CAIADO,2214,1.7,5
5,ROMEU ZEMA NETO,1277,1.0,6
6,SAMARA MARTINS DA SILVA FEITOSA,130,0.1,7
7,CLARIANA ZACARKIM BARAO,34,0.0,8
8,HERTZ DA CONCEICAO DIAS,31,0.0,9
9,EDMILSON SILVA COSTA,21,0.0,10


In [29]:
# Os 4 principais = os 4 mais votados na cidade
TOP4 = ranking_divi["NM_VOTAVEL"].head(4).tolist()

NOMES_CURTOS = {
    "FLAVIO NANTES BOLSONARO": "Flávio Bolsonaro",
    "LUIZ INÁCIO LULA DA SILVA": "Lula",
    "AUGUSTO JORGE CURY": "Augusto Cury",
    "RENAN ANTONIO FERREIRA DOS SANTOS": "Renan Santos",
}
print("TOP4:", [NOMES_CURTOS.get(n, n) for n in TOP4])

quarto, quinto = ranking_divi.iloc[3], ranking_divi.iloc[4]
print(f"4º: {quarto['NM_VOTAVEL']} ({quarto['QT_VOTOS']} votos) | "
      f"5º: {quinto['NM_VOTAVEL']} ({quinto['QT_VOTOS']} votos) | "
      f"diferença: {quarto['QT_VOTOS'] - quinto['QT_VOTOS']} votos")

TOP4: ['Flávio Bolsonaro', 'Lula', 'Augusto Cury', 'Renan Santos']
4º: RENAN ANTONIO FERREIRA DOS SANTOS (3219 votos) | 5º: RONALDO RAMOS CAIADO (2214 votos) | diferença: 1005 votos


### Votos por prédio 

Mesmas decisões do 2º turno: só as seções **principais** (as agregadas votam na urna da principal) e
**um ponto por coordenada** (prédios com mais de um código de local viram um ponto só).

In [30]:
por_secao_1t = votos_1t.pivot_table(
    index=["NR_ZONA", "NR_SECAO"], columns="NM_VOTAVEL", values="QT_VOTOS",
    aggfunc="sum", fill_value=0,
).reset_index()
por_secao_1t.columns.name = None
OPCOES_1T = [c for c in por_secao_1t.columns if c not in ("NR_ZONA", "NR_SECAO")]
CANDIDATOS_1T = [c for c in OPCOES_1T if c not in BRANCO_NULO]

# Cruzar com os locais 
for tabela in (por_secao_1t, locais_1t):
    for coluna in ("NR_ZONA", "NR_SECAO"):
        tabela[coluna] = tabela[coluna].str.strip()
locais_principais_1t = locais_1t[locais_1t["DS_TIPO_SECAO_AGREGADA"] == "Principal"]

secoes_1t = por_secao_1t.merge(
    locais_principais_1t[colunas_local], on=["NR_ZONA", "NR_SECAO"],
    how="left", validate="one_to_one",
)
print("seções com voto:", len(secoes_1t), "| seções sem lat/long:", secoes_1t["NR_LATITUDE"].isna().sum())

for coluna in ("NR_LATITUDE", "NR_LONGITUDE"):
    secoes_1t[coluna] = secoes_1t[coluna].str.replace(",", ".").astype(float)

# Agrupar por coordenada, somando todas as opções de voto
por_local_1t = secoes_1t.groupby(["NR_LATITUDE", "NR_LONGITUDE"], as_index=False).agg(
    NM_LOCAL_VOTACAO=("NM_LOCAL_VOTACAO", juntar_textos),
    NR_LOCAL_VOTACAO=("NR_LOCAL_VOTACAO", juntar_textos),
    NM_BAIRRO=("NM_BAIRRO", juntar_textos),
    NR_ZONA=("NR_ZONA", juntar_textos),
    QT_SECOES=("NR_SECAO", "count"),
    SECOES=("NR_SECAO", lambda s: ", ".join(sorted(s, key=int))),
    **{opcao: (opcao, "sum") for opcao in OPCOES_1T},
)
print("pontos no mapa:", len(por_local_1t))

seções com voto: 501 | seções sem lat/long: 0
pontos no mapa: 72


### Percentuais dos 4 principais e líder de cada prédio

Percentuais sobre os **votos válidos** do prédio (todos os 13 candidatos, sem brancos e nulos).
O **líder** é o candidato mais votado no prédio entre todos os 13 (não só entre os 4).

In [31]:
por_local_1t["VALIDOS"] = por_local_1t[CANDIDATOS_1T].sum(axis=1)
por_local_1t["TOTAL"] = por_local_1t[OPCOES_1T].sum(axis=1)

for candidato in TOP4:
    por_local_1t[f"PCT_{NOMES_CURTOS[candidato]}"] = 100 * por_local_1t[candidato] / por_local_1t["VALIDOS"]

por_local_1t["LIDER"] = por_local_1t[CANDIDATOS_1T].idxmax(axis=1).map(lambda n: NOMES_CURTOS.get(n, n))
por_local_1t["PCT_LIDER"] = 100 * por_local_1t[CANDIDATOS_1T].max(axis=1) / por_local_1t["VALIDOS"]

# Conferência 1: os totais por candidato têm que bater com o ranking da cidade
conferencia = por_local_1t[CANDIDATOS_1T].sum().sort_values(ascending=False)
print("totais batem com o ranking:",
      conferencia.astype(int).to_dict() == ranking_divi.set_index("NM_VOTAVEL")["QT_VOTOS"].to_dict())

# Conferência 2: empate no 1º lugar 
maximos = por_local_1t[CANDIDATOS_1T].max(axis=1)
empates_1t = (por_local_1t[CANDIDATOS_1T].eq(maximos, axis=0).sum(axis=1) > 1).sum()
print("prédios com empate no 1º lugar:", empates_1t)

print(por_local_1t["LIDER"].value_counts())
colunas_pct = [f"PCT_{NOMES_CURTOS[c]}" for c in TOP4]
por_local_1t[["NM_LOCAL_VOTACAO", "TOTAL", "LIDER"] + colunas_pct].round(1).head()

totais batem com o ranking: True
prédios com empate no 1º lugar: 0
LIDER
Flávio Bolsonaro    61
Lula                11
Name: count, dtype: int64


,NM_LOCAL_VOTACAO,TOTAL,LIDER,PCT_Flávio Bolsonaro,PCT_Lula,PCT_Augusto Cury,PCT_Renan Santos
0,ESCOLA MUNICIPAL BENJAMIM CONSTANT,707,Lula,33.9,56.5,3.3,2.5
1,CRAS - CENTRO DE REFERÊNCIA DE ASSISTÊNCIA SOCIAL,897,Lula,41.2,50.6,3.3,3.1
2,ESCOLA MUNICIPAL ANTONIETA FONSECA,782,Lula,36.9,51.5,5.3,3.4
3,ESCOLA MUNICIPAL DONA MARIA ROSA,951,Lula,43.0,47.5,4.3,2.9
4,ESCOLA MUNICIPAL ANTÔNIO PIO DA SILVA,329,Flávio Bolsonaro,57.0,28.7,5.7,4.3


## 1º turno – mapas de calor

Um mapa para cada um dos 4 principais candidatos, feito com o **`HeatMap`** do Folium. Cada local de
votação vira um circulo de cor, e circulos vizinhos se somam: o mapa mostra **em que regiões** o candidato foi forte,
sem precisar de fronteiras.

**O que esquenta o mapa: votos acima do esperado.** Em cada local, o *esperado* é quanto o candidato teria se o local
votasse igual à cidade: `votos válidos do local × % do candidato na cidade`. O peso do local é a diferença entre os
votos que ele teve e esse esperado. Exemplo: se o candidato teve 40% na cidade

In [32]:
# % de cada candidato nos votos válidos da cidade 
PCT_CIDADE_1T = ranking_divi.set_index("NM_VOTAVEL")["PCT_DIVI"]

for candidato in TOP4:
    nome = NOMES_CURTOS[candidato]
    esperado = por_local_1t["VALIDOS"] * PCT_CIDADE_1T[candidato] / 100
    por_local_1t[f"ACIMA_{nome}"] = por_local_1t[candidato] - esperado

# é distrito, povoado ou zona rural
PADRAO_FORA_DA_SEDE = "DISTRITO|POVOADO|ZONA RURAL"
FORA_DA_SEDE = por_local_1t["NM_BAIRRO"].str.contains(PADRAO_FORA_DA_SEDE)

print("soma dos votos acima do esperado dá zero para os 4:",
      all(abs(por_local_1t[f"ACIMA_{NOMES_CURTOS[c]}"].sum()) < 1e-6 for c in TOP4))

pd.DataFrame(
    {
        "% na cidade": [PCT_CIDADE_1T[c] for c in TOP4],
        "correlação votos × válidos do local": [por_local_1t[c].corr(por_local_1t["VALIDOS"]) for c in TOP4],
        "locais acima da média": [(por_local_1t[f"ACIMA_{NOMES_CURTOS[c]}"] > 0).sum() for c in TOP4],
        "maior saldo de um local (votos)": [por_local_1t[f"ACIMA_{NOMES_CURTOS[c]}"].max() for c in TOP4],
    },
    index=[NOMES_CURTOS[c] for c in TOP4],
).round(2)

soma dos votos acima do esperado dá zero para os 4: True


,% na cidade,correlação votos × válidos do local,locais acima da média,maior saldo de um local (votos)
Flávio Bolsonaro,53.98,0.98,35,425.11
Lula,37.31,0.96,41,192.43
Augusto Cury,3.45,0.95,35,37.13
Renan Santos,2.43,0.95,35,27.04


In [33]:
from folium.plugins import HeatMap

# Divinópolis tem duas zonas eleitorais (102 e 103)

NOMES_TOP4 = [NOMES_CURTOS[c] for c in TOP4]
PCT_CIDADE_CURTO = {NOMES_CURTOS[c]: PCT_CIDADE_1T[c] for c in TOP4}
PARTIDOS = {"Flávio Bolsonaro": "PL, 22", "Lula": "PT, 13", "Augusto Cury": "Avante, 70", "Renan Santos": "Missão, 14"}

DEGRADES = {
    "Flávio Bolsonaro": ["#c6dbef", "#6baed6", "#2171b5", "#08306b"],   # PL: azul
    "Lula":             ["#fcbba1", "#fb6a4a", "#cb181d", "#67000d"],   # PT: vermelho
    "Augusto Cury":     ["#fdd0a2", "#fd8d3c", "#d94801", "#7f2704"],   # Avante: laranja
    "Renan Santos":     ["#fff3a0", "#f5c400", "#8a6d00", "#1a1a1a"],   # Missão: amarelo -> preto
}

COR_NOME = {"Flávio Bolsonaro": "#2166ac", "Lula": "#b2182b", "Augusto Cury": "#d94801", "Renan Santos": "#8a6d00"}

RAIO_CALOR = 25          
DESFOQUE_CALOR = 15      
OPACIDADE_CALOR = 0.75  


def cartao_html_1t(local, candidato):
    """HTML do tooltip de um local no mapa de um candidato (nome curto)."""
    lider = local["LIDER"]
    linhas_tabela = "".join(
        f'<tr style="{"font-weight:700; background:#f2f2f2;" if nome == candidato else ""}">'
        f'<td style="color:{COR_NOME[nome]}; padding:1px 4px;">{nome}</td>'
        f'<td style="text-align:right; padding:1px 4px;">{pct_br(local[f"PCT_{nome}"])}</td></tr>'
        for nome in NOMES_TOP4
    )
    acima = int(round(local[f"ACIMA_{candidato}"]))
    if acima > 0:
        saldo = f"<b>{num_br(acima)} votos acima</b> do esperado"
    elif acima < 0:
        saldo = f"{num_br(-acima)} votos abaixo do esperado"
    else:
        saldo = "o esperado para a média da cidade"
    bairro = f'<div style="color:#666; font-size:12px; margin-bottom:8px;">Bairro: {local["NM_BAIRRO"]}</div>' if local["NM_BAIRRO"] else ""
    return f"""
    <div style="min-width:240px; max-width:320px; white-space:normal;">
      <div style="font-weight:700; font-size:14px; margin-bottom:2px;">{local["NM_LOCAL_VOTACAO"]}</div>
      <div style="color:#666; font-size:12px; margin-bottom:2px;">Zona {local["NR_ZONA"]} · Seções {local["SECOES"]}</div>
      {bairro}
      <div style="font-weight:700; margin-bottom:4px;">Líder: <span style="color:{COR_NOME.get(lider, '#222222')}">{lider}</span> com {pct_br(local["PCT_LIDER"])}</div>
      <div style="color:#666; font-size:11px;">% dos votos válidos no local:</div>
      <table style="width:100%; font-size:12px; border-collapse:collapse; margin-bottom:6px;">{linhas_tabela}</table>
      <div style="font-size:12px;">{candidato}: {pct_br(local[f"PCT_{candidato}"])} aqui, {pct_br(PCT_CIDADE_CURTO[candidato])} na cidade.<br>Teve {saldo}.</div>
      <div style="color:#666; font-size:12px; margin-top:6px;">{num_br(int(local["TOTAL"]))} votos ({num_br(int(local["TOTAL"] - local["VALIDOS"]))} brancos/nulos)</div>
    </div>"""


def legenda_calor_html(candidato):
    """Legenda: o degradê do calor na cor do partido + o que é cada ponto branco."""
    cores = DEGRADES[candidato]
    return f'''
    <div style="position:fixed; top:12px; right:12px; z-index:9999; width:300px;
                background:rgba(255,255,255,0.92); padding:8px 10px; border-radius:6px;
                box-shadow:0 1px 6px rgba(0,0,0,0.3); font-family:Arial, sans-serif; color:#222;">
      <div style="font-weight:700; font-size:13px;">1º turno – <span style="color:{COR_NOME[candidato]};">{candidato}</span> ({PARTIDOS[candidato]})</div>
      <div style="font-size:11px; margin:2px 0 6px;">Onde teve <b>mais votos do que o esperado</b> pela média da cidade
        ({pct_br(PCT_CIDADE_CURTO[candidato])} dos válidos)</div>
      <div style="height:12px; border:1px solid #999; opacity:{OPACIDADE_CALOR};
                  background:linear-gradient(to right, #ffffff, {", ".join(cores)});"></div>
      <div style="display:flex; justify-content:space-between; font-size:11px; margin-top:2px;">
        <span>pouco acima</span><span>muito acima</span>
      </div>
      <div style="display:flex; align-items:center; gap:6px; font-size:11px; margin-top:6px;">
        <div style="width:10px; height:10px; border-radius:50%; background:#ffffff; border:1px solid #555;"></div>
        Local de votação
      </div>
    </div>'''


def mapa_calor_1t(dados, candidato):
    """Mapa do 1º turno de um candidato (nome curto): calor dos votos acima do esperado + pontos com tooltip."""
    m = mapa_base()
    coluna = f"ACIMA_{candidato}"
    cores = DEGRADES[candidato]

    HeatMap(
        dados[["NR_LATITUDE", "NR_LONGITUDE", coluna]].values.tolist(),
        name="Calor: votos acima do esperado",
        radius=RAIO_CALOR,
        blur=DESFOQUE_CALOR,
        min_opacity=0.001,
        gradient={0.25: cores[0], 0.5: cores[1], 0.75: cores[2], 1.0: cores[3]},
    ).add_to(m)
    m.get_root().header.add_child(folium.Element(
        f"<style>.leaflet-heatmap-layer {{ opacity: {OPACIDADE_CALOR}; }}</style>"
    ))

    pontos = dados.copy()
    pontos["CARTAO"] = pontos.apply(cartao_html_1t, axis=1, candidato=candidato)
    gdf = gpd.GeoDataFrame(
        pontos[["CARTAO"]],
        geometry=gpd.points_from_xy(pontos["NR_LONGITUDE"], pontos["NR_LATITUDE"]),
        crs="EPSG:4326",
    )
    folium.GeoJson(
        gdf,
        name="Locais de votação",
        marker=folium.CircleMarker(),
        style_function=lambda f: {
            "radius": 5,
            "fill": True,
            "fillColor": "#ffffff",
            "fillOpacity": 0.9,
            "color": "#555555",
            "weight": 1,
        },
        highlight_function=lambda f: {"weight": 3, "color": "#000000"},
        tooltip=folium.GeoJsonTooltip(fields=["CARTAO"], labels=False, sticky=True, style=ESTILO_TOOLTIP),
    ).add_to(m)

    folium.LayerControl(position="bottomleft", collapsed=False).add_to(m)   
    m.get_root().html.add_child(folium.Element(legenda_calor_html(candidato)))
    na_sede = ~dados["NM_BAIRRO"].str.contains(PADRAO_FORA_DA_SEDE)
    oeste, sul, leste, norte = gdf[na_sede.values].total_bounds
    m.fit_bounds([[sul, oeste], [norte, leste]], max_zoom=13)
    return m

In [34]:
from pathlib import Path
Path("mapas").mkdir(exist_ok=True)

ARQUIVOS_1T = {"Flávio Bolsonaro": "flavio", "Lula": "lula", "Augusto Cury": "cury", "Renan Santos": "renan"}
mapas_1t = {}
for nome in NOMES_TOP4:
    mapas_1t[nome] = mapa_calor_1t(por_local_1t, nome)
    arquivo = f"mapas/mapa_1turno_{ARQUIVOS_1T[nome]}.html"
    mapas_1t[nome].save(arquivo)
    print("salvo:", arquivo)

salvo: mapas/mapa_1turno_flavio.html
salvo: mapas/mapa_1turno_lula.html
salvo: mapas/mapa_1turno_cury.html
salvo: mapas/mapa_1turno_renan.html


In [35]:
sede, fora = por_local_1t[~FORA_DA_SEDE], por_local_1t[FORA_DA_SEDE]
print(f"Locais na sede: {len(sede)}, com {pct_br(100 * sede['VALIDOS'].sum() / por_local_1t['VALIDOS'].sum())} dos válidos"
      f" | fora da sede: {len(fora)} ({', '.join(sorted(fora['NM_BAIRRO'].unique()))})")


def pct_em(tabela, candidato):
    """% do candidato nos votos válidos de um grupo de locais."""
    return 100 * tabela[candidato].sum() / tabela["VALIDOS"].sum()


zonas = {f"zona {z}": t for z, t in por_local_1t.groupby("NR_ZONA")}
display(pd.DataFrame(
    {
        "cidade": [pct_em(por_local_1t, c) for c in TOP4],
        **{nome: [pct_em(t, c) for c in TOP4] for nome, t in zonas.items()},
        "sede": [pct_em(sede, c) for c in TOP4],
        "fora da sede": [pct_em(fora, c) for c in TOP4],
    },
    index=NOMES_TOP4,
).round(1))

display(por_local_1t[[f"PCT_{n}" for n in NOMES_TOP4]].describe().loc[["min", "25%", "50%", "75%", "max"]].round(1))

for nome in NOMES_TOP4:
    saldo = por_local_1t.groupby("NM_BAIRRO")[f"ACIMA_{nome}"].sum()
    print(f"\n{nome} – mais acima: " + ", ".join(f"{b} ({v:+.0f})" for b, v in saldo.nlargest(4).items()))
    print(f"{nome} – mais abaixo: " + ", ".join(f"{b} ({v:+.0f})" for b, v in saldo.nsmallest(3).items()))

for nome in NOMES_TOP4:
    print(f"\n=== {nome} ===")
    print(por_local_1t.nlargest(3, f"ACIMA_{nome}")[["NM_LOCAL_VOTACAO", "NM_BAIRRO", "NR_ZONA", "VALIDOS", f"PCT_{nome}",
                                                     f"ACIMA_{nome}"]].round(1).to_string(index=False))
    for rotulo, i in (("menor %", por_local_1t[f"PCT_{nome}"].idxmin()), ("maior %", por_local_1t[f"PCT_{nome}"].idxmax())):
        r = por_local_1t.loc[i]
        print(f"{rotulo}: {r['NM_LOCAL_VOTACAO']} ({r['NM_BAIRRO']}, {r['VALIDOS']} válidos): {pct_br(r[f'PCT_{nome}'])}")


Locais na sede: 65, com 98,6% dos válidos | fora da sede: 7 (ZONA RURAL)


,cidade,zona 102,zona 103,sede,fora da sede
Flávio Bolsonaro,54.0,56.3,50.8,54.0,56.3
Lula,37.3,35.3,40.1,37.3,35.9
Augusto Cury,3.4,3.3,3.7,3.5,2.9
Renan Santos,2.4,2.3,2.6,2.4,1.7


,PCT_Flávio Bolsonaro,PCT_Lula,PCT_Augusto Cury,PCT_Renan Santos
min,33.9,24.0,0.9,0.0
25%,49.9,33.5,3.0,2.1
50%,53.7,38.1,3.4,2.4
75%,57.4,41.3,3.9,2.7
max,68.9,56.5,5.7,4.3



Flávio Bolsonaro – mais acima: CENTRO (+1411), SIDIL (+407), PLANALTO (+286), ALVORADA (+126)
Flávio Bolsonaro – mais abaixo: NITERÓI (-321), SERRA VERDE (-203), MARIA HELENA (-172)

Lula – mais acima: NITERÓI (+307), SERRA VERDE (+192), MARIA HELENA (+162), JARDIM DOS CANDIDÉS (+138)
Lula – mais abaixo: CENTRO (-1280), SIDIL (-425), PLANALTO (-269)

Augusto Cury – mais acima: DONA ROSA (+37), PARAÍSO (+24), OLIVEIRAS (+23), INTERLAGOS (+21)
Augusto Cury – mais abaixo: CENTRO (-139), PLANALTO (-17), ESPLANADA (-17)

Renan Santos – mais acima: SIDIL (+28), CENTRO (+24), ICARAÍ II (+14), ESPÍRITO SANTO (+14)
Renan Santos – mais abaixo: SANTO ANTÔNIO DOS CAMPOS (-29), PORTO VELHO (-24), SÃO SEBASTIÃO (-14)

=== Flávio Bolsonaro ===
                          NM_LOCAL_VOTACAO NM_BAIRRO NR_ZONA  VALIDOS  PCT_Flávio Bolsonaro  ACIMA_Flávio Bolsonaro
INSTITUTO NOSSA SENHORA DO SAGRADO CORAÇÃO    CENTRO     102     2847                  68.9                   425.1
                  COLÉGIO RO

In [36]:
mapas_1t["Flávio Bolsonaro"]

#### Relatório – Mapa do 1º turno: Flávio Bolsonaro

**O fato**
- **Venceu na cidade com 54,0% dos válidos** e liderou em 61 dos 72 locais.
- **O calor se concentra no Centro**, de longe o bairro com maior saldo, seguido de Sidil, Planalto e Alvorada. Os locais mais quentes são o Instituto Nossa Senhora do Sagrado Coração (68,9% dos válidos) e o Colégio Roberto Carneiro (66,2%), os dois no Centro, e a E.E. Manoel Correa Filho, no Planalto (61,6%).
- **O mapa fica sem cor em Niterói, Serra Verde e Maria Helena**, onde ele ficou abaixo da média: são
  os bairros onde aparece o calor do Lula.
- **Zonas e zona rural:** 56,3% na zona 102 e 50,8% na 103; 54,0% na sede e 56,3% nos 7 locais da zona rural.

**O design**
- **Paleta:** degradê azul do ColorBrewer (`Blues`). Azul é uma das cores do PL e é a cor do Flávio no mapa do 2º turno.

In [37]:
mapas_1t["Lula"]

#### Relatório – Mapa do 1º turno: Lula

**O fato**
- **Segundo lugar, com 37,3% dos válidos;** liderou em 11 dos 72 locais.
- **O mapa é o espelho do mapa do Flávio:** o calor fica em Niterói, Serra Verde, Maria Helena e Jardim dos Candidés. O local com maior saldo é o CAIC – E.M. Padre João Bruno, na Serra Verde (45,8% dos válidos); a E.M. Doutor Sebastião Gomes, no Maria Helena, deu 49,6% a ele.
- **O Centro fica sem cor**, assim como Sidil e Planalto, onde o Flávio esquenta.
- **Zonas e zona rural:** 40,1% na zona 103 e 35,3% na 102; 37,3% na sede e 35,9% na zona rural.

**O design**
- **Paleta:** degradê vermelho do ColorBrewer (`Reds`): a cor do PT e a mesma do Lula no mapa do 2º turno.

In [38]:
mapas_1t["Augusto Cury"]

#### Relatório – Mapa do 1º turno: Augusto Cury

**O fato**
- **Terceiro lugar, com 3,4% dos válidos, e um voto bem espalhado:** em metade dos locais ele teve entre 3,0% e 3,9%, e o
  maior saldo de um local é de só 37 votos.
- **O calor aparece em Dona Rosa, Paraíso, Oliveiras e Interlagos.** O local mais quente é a E.M.
  Professor Paulo Freire, em Dona Rosa (4,8%). 
- **O ponto fraco é o Centro**.
- **Zonas e zona rural:** 3,7% na zona 103 e 3,3% na 102; 3,5% na sede e 2,9% na zona rural.

**O design**
- **Paleta:** degradê laranja do ColorBrewer (`Oranges`): laranja é a cor principal do Avante.

In [39]:
mapas_1t["Renan Santos"]

#### Relatório – Mapa do 1º turno: Renan Santos

**O fato**
- **Quarto lugar, com 2,4% dos válidos** (3.219 votos), à frente de Ronaldo Caiado (1,7%).
- **O calor do Renan coincide com o do Flávio:** os bairros com maior saldo são Sidil e Centro, e os locais mais
  quentes são a E.E. Joaquim Nabuco, no Centro (3,6%), e a E.E. São Francisco de Assis, no Sidil (3,5%).
- **Fica abaixo da média em Santo Antônio dos Campos, Porto Velho e São Sebastião .**
- **Zonas e zona rural:** 2,6% na zona 103 e 2,3% na 102; 2,4% na sede e só 1,7% na zona rural.

**O design**
- **Paleta:** degradê amarelo → preto, as cores do Missão.

## 2º turno

### Passo 1 – Filtrar o 2º turno

Os dois arquivos trazem os dois turnos. Ficamos só com as linhas do 2º.

In [40]:
votos_2t = votos[votos["NR_TURNO"] == "2"].copy()
locais_2t = locais[locais["NR_TURNO"] == "2"].copy()

print("linhas de votos:", len(votos_2t), "| linhas de locais:", len(locais_2t))
votos_2t.groupby("NM_VOTAVEL")["QT_VOTOS"].sum()

linhas de votos: 1973 | linhas de locais: 504


NM_VOTAVEL
FLAVIO NANTES BOLSONARO      76540
LUIZ INÁCIO LULA DA SILVA    62494
VOTO BRANCO                   1827
VOTO NULO                     3211
Name: QT_VOTOS, dtype: int64

### Passo 2 – Pivotar: uma linha por seção

Hoje cada linha é "seção × candidato" (4 linhas por seção: Flávio, Lula, branco e nulo).
O `pivot_table` transforma isso em **uma linha por seção**, com uma coluna para cada opção de voto.

In [41]:
por_secao = votos_2t.pivot_table(
    index=["NR_ZONA", "NR_SECAO"],   
    columns="NM_VOTAVEL",            
    values="QT_VOTOS",               
    aggfunc="sum",
    fill_value=0,                    
).reset_index()
por_secao.columns.name = None

# Nomes curtos para as colunas
por_secao = por_secao.rename(columns={
    "FLAVIO NANTES BOLSONARO": "FLAVIO",
    "LUIZ INÁCIO LULA DA SILVA": "LULA",
    "VOTO BRANCO": "BRANCO",
    "VOTO NULO": "NULO",
})

print("seções com votos:", len(por_secao))
por_secao.head()

seções com votos: 501


,NR_ZONA,NR_SECAO,FLAVIO,LULA,BRANCO,NULO
0,102,1,207,85,3,2
1,102,10,180,110,2,2
2,102,100,147,132,0,6
3,102,101,138,145,3,7
4,102,102,160,125,5,3


### Passo 3 – Cruzar votos com locais (chave: `NR_ZONA` + `NR_SECAO`)

A tabela de votos não tem latitude/longitude e a de locais não tem votos. Juntamos as duas pela chave **zona + seção**.

**Padronização das chaves:** as duas colunas precisam estar no mesmo formato nas duas tabelas
(texto sem espaços). Se uma tivesse `"69"` e a outra `69` ou `"069"`, o `merge` não encontraria o par.

**Seções agregadas:** no arquivo de locais há seções do tipo *Agregada*. Os eleitores delas votam na urna
de outra seção (a *Principal*), então elas não têm votos próprios: os votos já estão somados na principal.
Usamos só as principais para não contar nada duas vezes.

In [42]:
# Padronizando as chaves
for tabela in (por_secao, locais_2t):
    for coluna in ("NR_ZONA", "NR_SECAO"):
        tabela[coluna] = tabela[coluna].str.strip()

print(locais_2t["DS_TIPO_SECAO_AGREGADA"].value_counts())

locais_principais = locais_2t[locais_2t["DS_TIPO_SECAO_AGREGADA"] == "Principal"]

secoes = por_secao.merge(
    locais_principais[colunas_local],
    on=["NR_ZONA", "NR_SECAO"],
    how="left",                 
    validate="one_to_one",     
)

print("seções sem lat/long:", secoes["NR_LATITUDE"].isna().sum())
secoes.head()

DS_TIPO_SECAO_AGREGADA
Principal    501
Agregada       3
Name: count, dtype: int64
seções sem lat/long: 0


,NR_ZONA,NR_SECAO,FLAVIO,LULA,BRANCO,NULO,NR_LOCAL_VOTACAO,NM_LOCAL_VOTACAO,DS_ENDERECO,NM_BAIRRO,NR_LATITUDE,NR_LONGITUDE
0,102,1,207,85,3,2,1503,COLÉGIO ROBERTO CARNEIRO,"RUA MINAS GERAIS, 593",CENTRO,"-20,1448072","-44,8912486"
1,102,10,180,110,2,2,1066,COLÉGIO APOGEU - ANTIGA FACED,"PRAÇA DO MERCADO MUNICIPAL, 191",CENTRO,"-20,1392271","-44,8880672"
2,102,100,147,132,0,6,1228,ESCOLA ESTADUAL MARTIN CYPRIEN - POLIVALENTE,"RUA AMAZONAS, 881",SÃO JOSÉ,"-20,158126","-44,9025969"
3,102,101,138,145,3,7,1228,ESCOLA ESTADUAL MARTIN CYPRIEN - POLIVALENTE,"RUA AMAZONAS, 881",SÃO JOSÉ,"-20,158126","-44,9025969"
4,102,102,160,125,5,3,1228,ESCOLA ESTADUAL MARTIN CYPRIEN - POLIVALENTE,"RUA AMAZONAS, 881",SÃO JOSÉ,"-20,158126","-44,9025969"


O TSE escreve as coordenadas com vírgula decimal (no formato `-xx,xxxxxxx`). Convertemos para número com ponto.

In [43]:
for coluna in ("NR_LATITUDE", "NR_LONGITUDE"):
    secoes[coluna] = secoes[coluna].str.replace(",", ".").astype(float)

secoes[["NR_LATITUDE", "NR_LONGITUDE"]].describe()

,NR_LATITUDE,NR_LONGITUDE
count,501.000000,501.000000
mean,-20.143574,-44.891070
std,0.024071,0.027435
min,-20.250716,-45.108838
25%,-20.158126,-44.902597
50%,-20.144831,-44.889588
75%,-20.134297,-44.874987
max,-19.993703,-44.798721


### Passo 4 – Agrupar por ponto no mapa (coordenada)

Várias seções funcionam no mesmo prédio e têm **a mesma coordenada**. Se cada seção virasse um círculo,
eles ficariam empilhados no mapa.

Primeiro tentamos agrupar pelo **código do local** (`NR_LOCAL_VOTACAO`), mas o mapa ainda mostrou círculos
um dentro do outro: alguns prédios aparecem no TSE com **dois códigos de local diferentes e a mesma coordenada**
(ex.: UEMG – Universidade do Estado de Minas Gerais; Colégio São José; E.E. Padre Matias Lobato; E.M. CETEPE). Por isso agrupamos pela **coordenada**: um círculo por ponto do mapa, somando todos os votos
daquele lugar. Os nomes e as seções de cada ponto ficam guardados para o tooltip.

Em Divinópolis há mais um motivo para não usar o código: o número do local **se repete entre as zonas 102 e 103**
(o 1082 é uma escola em cada zona). Um local só é identificado por **zona + código**: são 76, em 72 pontos do mapa.

In [44]:
por_local = secoes.groupby(["NR_LATITUDE", "NR_LONGITUDE"], as_index=False).agg(
    NM_LOCAL_VOTACAO=("NM_LOCAL_VOTACAO", juntar_textos),
    NR_LOCAL_VOTACAO=("NR_LOCAL_VOTACAO", juntar_textos),
    NM_BAIRRO=("NM_BAIRRO", juntar_textos),
    DS_ENDERECO=("DS_ENDERECO", "first"),
    FLAVIO=("FLAVIO", "sum"),
    LULA=("LULA", "sum"),
    BRANCO=("BRANCO", "sum"),
    NULO=("NULO", "sum"),
    NR_ZONA=("NR_ZONA", juntar_textos),
    QT_SECOES=("NR_SECAO", "count"),
    SECOES=("NR_SECAO", lambda s: ", ".join(sorted(s, key=int))),
)

print("pontos no mapa:", len(por_local))
print("locais diferentes (zona + código):", secoes[["NR_ZONA", "NR_LOCAL_VOTACAO"]].drop_duplicates().shape[0])

# Os pontos que juntaram mais de um código de local (vale citar no relatório)
por_local[por_local["NR_LOCAL_VOTACAO"].str.contains("/")][["NM_LOCAL_VOTACAO", "NR_LOCAL_VOTACAO", "SECOES"]]

pontos no mapa: 72
locais diferentes (zona + código): 76


,NM_LOCAL_VOTACAO,NR_LOCAL_VOTACAO,SECOES
7,UEMG - UNIVERSIDADE DO ESTADO DE MINAS GERAIS,1520 / 1350,"103, 104, 105, 157, 166, 176, 184, 193, 202, 2..."
27,COLÉGIO SÃO JOSÉ,1210 / 1112,"77, 78, 79, 80, 81, 82, 83, 84"
38,ESCOLA ESTADUAL PADRE MATIAS LOBATO,1082 / 1155,"5, 6, 7, 17, 18, 19, 27, 28, 29, 30, 206"
52,ESCOLA MUNICIPAL CETEPE,1260 / 1406,"129, 130, 131, 132, 133, 134, 180, 188, 195, 2..."


### Passo 5 – Percentuais sobre os votos válidos

Votos válidos = votos nos candidatos, sem brancos e nulos. É o critério oficial do TSE para definir o vencedor,
e com ele `% Flávio + % Lula = 100%`, então **50% é exatamente o empate**: o ponto neutro da paleta divergente.

In [45]:
por_local["VALIDOS"] = por_local["FLAVIO"] + por_local["LULA"]
por_local["TOTAL"] = por_local["VALIDOS"] + por_local["BRANCO"] + por_local["NULO"]
por_local["PCT_FLAVIO"] = 100 * por_local["FLAVIO"] / por_local["VALIDOS"]
por_local["PCT_LULA"] = 100 * por_local["LULA"] / por_local["VALIDOS"]

print(por_local[["FLAVIO", "LULA", "BRANCO", "NULO"]].sum())

por_local.sort_values("PCT_LULA").head(10)

FLAVIO    76540
LULA      62494
BRANCO     1827
NULO       3211
dtype: int64


,NR_LATITUDE,NR_LONGITUDE,NM_LOCAL_VOTACAO,NR_LOCAL_VOTACAO,NM_BAIRRO,DS_ENDERECO,FLAVIO,LULA,BRANCO,NULO,NR_ZONA,QT_SECOES,SECOES,VALIDOS,TOTAL,PCT_FLAVIO,PCT_LULA
40,-20.141402,-44.891189,INSTITUTO NOSSA SENHORA DO SAGRADO CORAÇÃO,1104,CENTRO,"AVENIDA SETE DE SETEMBRO, 1180",2058,842,23,41,102,10,"22, 23, 24, 25, 149, 174, 207, 242, 270, 301",2900,2964,70.965517,29.034483
33,-20.144807,-44.891249,COLÉGIO ROBERTO CARNEIRO,1503,CENTRO,"RUA MINAS GERAIS, 593",2038,918,20,31,102,10,"1, 2, 15, 16, 181, 183, 199, 203, 215, 216",2956,3007,68.944520,31.055480
70,-20.020019,-44.952470,CENTRO COMUNITÁRIO DE QUILOMBO,1317,ZONA RURAL,COMUNIDADE RURAL DE QUILOMBO,198,95,2,6,102,1,143,293,301,67.576792,32.423208
45,-20.136816,-44.899167,ESCOLA ESTADUAL SÃO FRANCISCO DE ASSIS,1309,SIDIL,"RUA ITAPECERICA, 1495",1438,797,27,36,102,8,"87, 88, 151, 156, 186, 208, 232, 278",2235,2298,64.340045,35.659955
41,-20.140995,-44.898682,COLÉGIO SANTO AGOSTINHO - UNIDADE COLÉGIO INTE...,1198,SIDIL,"RUA SÃO PAULO, 1441",1935,1095,16,59,102,10,"64, 65, 66, 67, 68, 153, 162, 243, 269, 298",3030,3105,63.861386,36.138614
28,-20.146654,-44.907867,ESCOLA ESTADUAL MANOEL CORREA FILHO,1384,PLANALTO,"RUA CAPITÓLIO, 280",2513,1444,43,59,102,12,"160, 168, 178, 189, 201, 213, 225, 236, 261, 2...",3957,4059,63.507708,36.492292
38,-20.142633,-44.891134,ESCOLA ESTADUAL PADRE MATIAS LOBATO,1082 / 1155,CENTRO,"PRAÇA BENEDITO VALADARES , S/N",1761,1048,34,39,102,11,"5, 6, 7, 17, 18, 19, 27, 28, 29, 30, 206",2809,2882,62.691349,37.308651
34,-20.144511,-44.901987,CENTRO PASTORAL DA IGREJA NOSSA SENHORA DA GUIA,1554,VILA BELO HORIZONTE,"RUA SERGIPE, 1250",533,319,6,12,102,3,"38, 39, 40",852,870,62.558685,37.441315
43,-20.139227,-44.888067,COLÉGIO APOGEU - ANTIGA FACED,1066,CENTRO,"PRAÇA DO MERCADO MUNICIPAL, 191",1527,919,17,35,102,9,"8, 9, 10, 11, 12, 13, 14, 20, 21",2446,2498,62.428455,37.571545
59,-20.113357,-44.866961,ESCOLA ESTADUAL ROSA VAZ DE ARAÚJO,1260,VILA ROMANA,"RUA LEÃO XIII,140",718,442,12,19,103,4,"140, 173, 217, 252",1160,1191,61.896552,38.103448


### Passo 6 – Mapa do 2º turno

- Um círculo por ponto do mapa, com **área proporcional ao total de votos**.
- **Paleta divergente vermelho ↔ azul**: vermelho = Lula (13, PT), azul = Flávio (22, PL).
- A cor representa o **% de Lula sobre os votos válidos**. O branco/neutro fica exatamente em **50% (empate)**.

In [46]:
import branca.colormap as cm

RDBU_6 = ["#2166ac", "#67a9cf", "#d1e5f0", "#fddbc7", "#ef8a62", "#b2182b"]
RDBU_8 = ["#2166ac", "#4393c3", "#92c5de", "#d1e5f0", "#fddbc7", "#f4a582", "#d6604d", "#b2182b"]
VERMELHO_LULA, AZUL_FLAVIO = "#b2182b", "#2166ac"   

BORDAS_A = [0, 30, 40, 50, 60, 70, 100]         
BORDAS_B = [10, 20, 30, 40, 50, 60, 70, 80, 90]  


def raio(total, total_max, raio_max=20):
    """Raio em pixels com ÁREA proporcional aos votos (por isso a raiz quadrada).
    Se o raio fosse proporcional aos votos, um local com o dobro de votos teria 4x a área
    e pareceria muito maior do que é."""
    return 3 + (raio_max - 3) * np.sqrt(total / total_max)


def cartao_html(local):
    """Monta o HTML do tooltip de um ponto do mapa."""
    if local["LULA"] > local["FLAVIO"]:
        lider = f'<span style="color:{VERMELHO_LULA}">Lula</span> lidera com {pct_br(local["PCT_LULA"])}'
    elif local["FLAVIO"] > local["LULA"]:
        lider = f'<span style="color:{AZUL_FLAVIO}">Flávio</span> lidera com {pct_br(local["PCT_FLAVIO"])}'
    else:
        lider = "Empate"

    bairro = f'<div style="color:#666; font-size:12px; margin-bottom:8px;">Bairro: {local["NM_BAIRRO"]}</div>' if local["NM_BAIRRO"] else ""
    return f"""
    <div style="min-width:240px; max-width:320px; white-space:normal;">
      <div style="font-weight:700; font-size:14px; margin-bottom:2px;">{local["NM_LOCAL_VOTACAO"]}</div>
      <div style="color:#666; font-size:12px; margin-bottom:2px;">Zona {local["NR_ZONA"]} · Seções {local["SECOES"]}</div>
      {bairro}
      <div style="font-weight:700; margin-bottom:4px;">{lider}</div>
      <div style="display:flex; height:10px; border-radius:5px; overflow:hidden; margin-bottom:4px;">
        <div style="width:{local["PCT_LULA"]:.1f}%; background:{VERMELHO_LULA};"></div>
        <div style="width:{local["PCT_FLAVIO"]:.1f}%; background:{AZUL_FLAVIO};"></div>
      </div>
      <div style="display:flex; justify-content:space-between; font-size:12px;">
        <span style="color:{VERMELHO_LULA}">Lula {pct_br(local["PCT_LULA"])}</span>
        <span style="color:{AZUL_FLAVIO}">Flávio {pct_br(local["PCT_FLAVIO"])}</span>
      </div>
      <div style="color:#666; font-size:12px; margin-top:6px;">{num_br(local["TOTAL"])} votos ({num_br(local["BRANCO"] + local["NULO"])} brancos/nulos)</div>
    </div>"""


def rotulos_classes(bordas):
    """[0, 30, 40, ..., 100] -> ['até 30', '30–40', ..., '70 ou mais']"""
    rotulos = []
    for i, (inicio, fim) in enumerate(zip(bordas[:-1], bordas[1:])):
        if i == 0 and inicio == 0:
            rotulos.append(f"até {fim}")
        elif i == len(bordas) - 2 and fim == 100:
            rotulos.append(f"{inicio} ou mais")
        else:
            rotulos.append(f"{inicio}–{fim}")
    return rotulos


def legenda_html(bordas, cores, titulo, cor_empate=None):
    """Legenda com todas as classes do mesmo tamanho (a legenda padrão do branca é proporcional ao valor)."""
    caixas = "".join(
        f'''<div style="flex:1; text-align:center;">
             <div style="height:14px; background:{cor}; border:1px solid #999;"></div>
             <div style="font-size:11px; margin-top:2px;">{rotulo}</div>
           </div>'''
        for cor, rotulo in zip(cores, rotulos_classes(bordas))
    )
    caixa_empate = ""
    if cor_empate:
        caixa_empate = f'''
      <div style="display:flex; align-items:center; gap:6px; font-size:11px; margin-top:6px;">
        <div style="width:14px; height:14px; background:{cor_empate}; border:1px solid #999;"></div>
        Empate exato (mesmo número de votos)
      </div>'''
    return f'''
    <div style="position:fixed; top:12px; right:12px; z-index:9999; width:340px;
                background:rgba(255,255,255,0.92); padding:8px 10px; border-radius:6px;
                box-shadow:0 1px 6px rgba(0,0,0,0.3); font-family:Arial, sans-serif; color:#222;">
      <div style="font-weight:700; font-size:12px; margin-bottom:6px;">{titulo}</div>
      <div style="display:flex; gap:2px;">{caixas}</div>{caixa_empate}
      <div style="display:flex; justify-content:space-between; font-size:11px; margin-top:4px;">
        <span style="color:{AZUL_FLAVIO};">&#9664; Flávio venceu</span>
        <span style="color:{VERMELHO_LULA};">Lula venceu &#9654;</span>
      </div>
    </div>'''


def mapa_2turno(dados, bordas, cores, titulo, cor_empate="#ffffff"):
    escala = cm.StepColormap(
        colors=cores,
        index=bordas,
        vmin=bordas[0],
        vmax=bordas[-1],
    )   

    pontos = dados.copy()
    pontos["RAIO"] = raio(pontos["TOTAL"], pontos["TOTAL"].max())
    pontos["PCT_COR"] = pontos["PCT_LULA"].clip(bordas[0], bordas[-1] - 1e-9)
    empate = pontos["LULA"] == pontos["FLAVIO"]
    pontos["COR"] = pontos["PCT_COR"].apply(escala)
    pontos.loc[empate, "COR"] = cor_empate
    pontos["CARTAO"] = pontos.apply(cartao_html, axis=1)
    pontos = pontos.sort_values("TOTAL", ascending=False)

    gdf = gpd.GeoDataFrame(
        pontos[["CARTAO", "RAIO", "COR"]],
        geometry=gpd.points_from_xy(pontos["NR_LONGITUDE"], pontos["NR_LATITUDE"]),
        crs="EPSG:4326",
    )

    m = mapa_base()

    folium.GeoJson(
        gdf,
        name="2º turno",
        marker=folium.CircleMarker(),  
        style_function=lambda f: {
            "radius": f["properties"]["RAIO"],
            "fill": True,               
            "fillColor": f["properties"]["COR"],
            "fillOpacity": 0.85,
            "color": "#555555",        
            "weight": 0.8,
        },
        highlight_function=lambda f: {"weight": 3, "color": "#000000"},   
        tooltip=folium.GeoJsonTooltip(
            fields=["CARTAO"],
            labels=False,              
            sticky=True,
            style=ESTILO_TOOLTIP,
        ),
    ).add_to(m)

    m.get_root().html.add_child(folium.Element(legenda_html(bordas, cores, titulo, cor_empate if empate.any() else None)))  
    oeste, sul, leste, norte = gdf.total_bounds
    m.fit_bounds([[sul, oeste], [norte, leste]])
    return m

In [47]:
print("Opção A:")
print(pd.cut(por_local["PCT_LULA"], BORDAS_A).value_counts().sort_index())
print()
print("Opção B:")
print(pd.cut(por_local["PCT_LULA"].clip(10, 90), BORDAS_B, include_lowest=True).value_counts().sort_index())

Opção A:
PCT_LULA
(0, 30]       1
(30, 40]     14
(40, 50]     41
(50, 60]     12
(60, 70]      4
(70, 100]     0
Name: count, dtype: int64

Opção B:
PCT_LULA
(9.999, 20.0]     0
(20.0, 30.0]      1
(30.0, 40.0]     14
(40.0, 50.0]     41
(50.0, 60.0]     12
(60.0, 70.0]      4
(70.0, 80.0]      0
(80.0, 90.0]      0
Name: count, dtype: int64


In [48]:
from pathlib import Path
Path("mapas").mkdir(exist_ok=True)

mapa_2t = mapa_2turno(por_local, BORDAS_A, RDBU_6, "2º turno – % dos votos válidos em Lula")
mapa_2t.save("mapas/mapa_2turno.html")
mapa_2t

In [49]:
flavio, lula = por_local["FLAVIO"].sum(), por_local["LULA"].sum()
validos, total = flavio + lula, por_local["TOTAL"].sum()
print(f"Flávio {pct_br(100 * flavio / validos)} × Lula {pct_br(100 * lula / validos)} dos válidos | "
      f"diferença: {num_br(int(flavio - lula))} votos | brancos e nulos: {pct_br(100 * (total - validos) / total)}")

por_local["VENCEDOR"] = np.select(
    [por_local["LULA"] > por_local["FLAVIO"], por_local["FLAVIO"] > por_local["LULA"]],
    ["Lula", "Flávio Bolsonaro"],
    default="Empate",
)
comparacao = por_local_1t[["NR_LATITUDE", "NR_LONGITUDE", "LIDER"]].merge(
    por_local[["NR_LATITUDE", "NR_LONGITUDE", "VENCEDOR"]], on=["NR_LATITUDE", "NR_LONGITUDE"], validate="one_to_one"
)
display(pd.crosstab(comparacao["LIDER"], comparacao["VENCEDOR"], margins=True, margins_name="total"))
print("empate:", por_local.loc[por_local["VENCEDOR"] == "Empate", "NM_LOCAL_VOTACAO"].tolist())
print("mediana de votos dos locais – onde o Lula venceu:", por_local.loc[por_local["VENCEDOR"] == "Lula", "TOTAL"].median(),
      "| onde o Flávio venceu:", por_local.loc[por_local["VENCEDOR"] == "Flávio Bolsonaro", "TOTAL"].median())

fora_2t = por_local["NM_BAIRRO"].str.contains(PADRAO_FORA_DA_SEDE)
grupos = {f"zona {z}": t for z, t in por_local.groupby("NR_ZONA")}
grupos.update({"sede": por_local[~fora_2t], "fora da sede": por_local[fora_2t]})
print("% do Lula:", {nome: pct_br(100 * t["LULA"].sum() / t["VALIDOS"].sum()) for nome, t in grupos.items()})

bairros = por_local.groupby("NM_BAIRRO")[["FLAVIO", "LULA", "VALIDOS"]].sum()
bairros["PCT_LULA"] = 100 * bairros["LULA"] / bairros["VALIDOS"]
grandes = bairros[bairros["VALIDOS"] > 1500].sort_values("PCT_LULA")
print("mais Flávio:", ", ".join(f"{b} ({pct_br(100 - v)})" for b, v in grandes["PCT_LULA"].head(4).items()))
print("mais Lula:", ", ".join(f"{b} ({pct_br(v)})" for b, v in grandes["PCT_LULA"].tail(4)[::-1].items()))

for rotulo, i in (("menor % do Lula", por_local["PCT_LULA"].idxmin()), ("maior % do Lula", por_local["PCT_LULA"].idxmax())):
    r = por_local.loc[i]
    print(f"{rotulo}: {r['NM_LOCAL_VOTACAO']} ({r['NM_BAIRRO']}, {r['VALIDOS']} válidos): {pct_br(r['PCT_LULA'])}")
print(por_local[por_local["NM_LOCAL_VOTACAO"].str.contains("UFSJ|UNIVERSIDADE|CAMPUS")]
      [["NM_LOCAL_VOTACAO", "NM_BAIRRO", "VALIDOS", "PCT_LULA"]].round(1).to_string(index=False))

Flávio 55,1% × Lula 44,9% dos válidos | diferença: 14.046 votos | brancos e nulos: 3,5%


VENCEDOR,Empate,Flávio Bolsonaro,Lula,total
LIDER,,,,
Flávio Bolsonaro,1,55,5,61
Lula,0,0,11,11
total,1,55,16,72


empate: ['ESCOLA ESTADUAL SÃO VICENTE']
mediana de votos dos locais – onde o Lula venceu: 1104.0 | onde o Flávio venceu: 2482.0
% do Lula: {'zona 102': '42,5%', 'zona 103': '48,4%', 'sede': '45,0%', 'fora da sede': '44,2%'}
mais Flávio: SIDIL (64,1%), PLANALTO (63,5%), CENTRO (63,1%), ALVORADA (61,3%)
mais Lula: SERRA VERDE (53,5%), NITERÓI (51,6%), PARAÍSO (51,4%), DAVANUZE (50,9%)
menor % do Lula: INSTITUTO NOSSA SENHORA DO SAGRADO CORAÇÃO (CENTRO, 2900 válidos): 29,0%
maior % do Lula: ESCOLA MUNICIPAL BENJAMIM CONSTANT (BURITIS, 696 válidos): 64,8%
                             NM_LOCAL_VOTACAO               NM_BAIRRO  VALIDOS  PCT_LULA
UEMG - UNIVERSIDADE DO ESTADO DE MINAS GERAIS JARDIM BELVEDERE I E II     3750      43.7


#### Relatório – Mapa do 2º turno

**O fato**
- **Flávio venceu com 55,1% dos válidos, contra 44,9% do Lula:** diferença de 14.046 votos.
- **Do 1º para o 2º turno, quem mais cresceu foi o Lula:** de 37,3% para 44,9% dos válidos. Nenhum local passou do Lula para o Flávio.
- **Flávio venceu em 55 dos 72 locais e Lula em 16**
- **Padrão espacial:** entre os bairros com mais de 1.500 válidos, o Flávio foi mais forte em Sidil (64,1%), Planalto
  (63,5%), Centro (63,1%) e Alvorada (61,3%); o Lula venceu em Serra Verde (53,5%), Niterói (51,6%), Paraíso (51,4%) e
  Davanuze (50,9%).
- **Zonas, sede e zona rural:** o Lula teve 48,4% na zona 103 e 42,5% na 102; sede e zona rural ficaram parecidas (45,0% ×
  44,2%).
- **Extremos:** o Lula teve 29,0% no Instituto Nossa Senhora do Sagrado Coração (Centro) e 64,8% na E.M. Benjamim Constant
  (Buritis). 

**O design**
- **Paleta divergente RdBu** (vermelho = Lula/PT, azul = Flávio/PL), com o centro neutro fixado em 50% (empate). A escala automática colocaria o centro em 46,9%.
- **Empate exato em branco**.
- **Círculos** (um por prédio) com **área proporcional ao total de votos**.

## Tabelas para o dashboard

In [50]:
Path("resultados").mkdir(exist_ok=True)

# Ranking do 1º turno
ranking_divi.assign(NOME=ranking_divi["NM_VOTAVEL"].map(lambda n: NOMES_CURTOS.get(n, n))).to_csv(
    "resultados/ranking_1turno.csv", sep=";", index=False, encoding="utf-8"
)

identificacao = ["NM_LOCAL_VOTACAO", "NM_BAIRRO", "NR_ZONA", "SECOES", "NR_LATITUDE", "NR_LONGITUDE"]

locais_1t = por_local_1t[identificacao + ["VALIDOS", "TOTAL", "LIDER", "PCT_LIDER"]].copy()
locais_1t["FORA_DA_SEDE"] = FORA_DA_SEDE
for candidato in TOP4:
    nome = NOMES_CURTOS[candidato]
    locais_1t[f"VOTOS_{nome}"] = por_local_1t[candidato]
    locais_1t[f"PCT_{nome}"] = por_local_1t[f"PCT_{nome}"]
    locais_1t[f"ACIMA_{nome}"] = por_local_1t[f"ACIMA_{nome}"]
locais_1t.to_csv("resultados/locais_1turno.csv", sep=";", index=False, encoding="utf-8")

locais_2t = por_local[identificacao + ["FLAVIO", "LULA", "BRANCO", "NULO", "VALIDOS", "TOTAL",
                                       "PCT_FLAVIO", "PCT_LULA", "VENCEDOR"]].copy()
locais_2t["FORA_DA_SEDE"] = por_local["NM_BAIRRO"].str.contains(PADRAO_FORA_DA_SEDE)
locais_2t.to_csv("resultados/locais_2turno.csv", sep=";", index=False, encoding="utf-8")

print("salvo em resultados/:", sorted(arquivo.name for arquivo in Path("resultados").iterdir()))

salvo em resultados/: ['locais_1turno.csv', 'locais_2turno.csv', 'ranking_1turno.csv']
